# Medical multiple-choice benchmark: public reproduction

Open this notebook in VS Code or JupyterLab using the environment installed from `requirements.lock`, then restart the kernel and Run All. It reads the structured public reference and uses the exact frozen presentation source for Main Figures 1-3 and the five Extended Data figures.

The public data contain saved outcomes and numerical estimates. No hosted-model calls or statistical recomputation occur in this notebook. Exports are enabled to a fresh `analysis/output/public-render-v01/`. To recalculate the scientific results from per-question records, follow the separate explicit command in the README with your locally acquired upstream MedQA test file.


# Study and reproduction route

The study comprises 1,273 MedQA questions, 15 generative configurations and 13 decision models. Each generative configuration has one run; each decision model has three. The public structured derivative contains 68,742 outcomes and the scientific fields from 69,318 initial/retry calls. Original first-attempt outcomes are retained. Raw requests, responses and reasoning text remain private. Questions, options and reference keys are obtained from upstream MedQA.


## 1. Settings

Use `explore` for normal interactive work. Use `render` only after selecting a saved result file and its SHA-256 below. It displays retained saved results under the current report scope and identifies the original run/date/SAP before any result tables. Additional provider-scalar analyses are excluded from this view; the original bundle and its bindings remain unchanged. Historical bundles remain available for deliberate review. `compute` is reserved for the final approved snapshot. Normal exploration needs no environment variables.


In [ ]:
import sys, platform
from pathlib import Path
from importlib.metadata import version
import pandas as pd
from IPython.display import display, Markdown

sys.dont_write_bytecode = True
working_directory = Path.cwd().resolve()
REPOSITORY = next((p for p in (working_directory, *working_directory.parents)
                   if (p / "provenance/data_manifest.json").is_file()), None)
if REPOSITORY is None:
    raise ValueError("Open this notebook from the public repository")
if str(REPOSITORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY))
from analysis.support.notebook import model_labels, repeatability_tables
from analysis.public_release import load_public_results, export_public_report

MODE = "render"
SHOW_SECONDARY = True
SHOW_DETAILS = False
DRAW_FIGURES = True
SAP_VERSION = "1.8"
RESULT_JSON = REPOSITORY / "data/reference_results.json.gz"
RESULT_SHA256 = "49003cb67c1d2f179dd71f81a10df4de6e51f8e917066ca8b07c6e3b59ca77f9"
summary, saved_result, review_tables = None, None, None
figures = {}
print("Working mode:", MODE)


## 2. Input integrity

Verify the preserved historical input and each explicitly selected extension package before calculation. The expanded design requires 68,742 resolved outcomes across G01–G15 and D01–D13. Missing models, runs and unresolved recovery remain visible. Incomplete coverage blocks S1 and final export. Cost and latency use the same selected final attempt; donor counts and missing values appear before calculation.


In [ ]:
saved_result = load_public_results(REPOSITORY)
summary = saved_result["summary"]
review_tables = {"models": saved_result["models"]}
coverage = summary["coverage"]
display(Markdown("**Public saved-result presentation.** The public derivative is bound to the accepted numerical result and verified presentation. Scientific values and figure settings are retained; private source metadata and benchmark text are omitted."))
display(pd.DataFrame([
    {"Record": "Source numerical result SHA-256", "Value": saved_result["source_result_sha256"]},
    {"Record": "Public reference SHA-256", "Value": RESULT_SHA256},
    {"Record": "SAP version", "Value": SAP_VERSION},
    {"Record": "Question/model/run outcomes", "Value": coverage["terminal"]},
]))


In [ ]:
MODEL_NAMES = model_labels(review_tables["models"])

def ci(values, scale=1):
    return "Unavailable" if values is None else f"{values[0]*scale:.2f}–{values[1]*scale:.2f}"

def p_value(value):
    return "Unavailable" if value is None else "<0.001" if value < .001 else f"{value:.3f}"

def paired_table(rows):
    return pd.DataFrame([{
        "Model A": MODEL_NAMES[r["positive_configuration"]],
        "Model B": MODEL_NAMES[r["negative_configuration"]],
        "Matched n": r["n"], "Both correct": r["both_correct"],
        "Only A correct": r["positive_discordant"], "Only B correct": r["negative_discordant"],
        "Both incorrect": r["both_incorrect"], "Newcombe 95% (pp)": ci(r.get("newcombe95_pp")), "A − B (percentage points)": r["difference_pp"],
        "95% bootstrap interval": ci(r["interval95_pp"]),
        **({"Raw P": p_value(r["p_exact"]), "Holm P": p_value(r.get("p_holm"))}
           if "p_exact" in r else {}),
    } for r in rows])

if MODE in ("explore", "compute"):
    integrity_rows = resource_integrity(manifest, terminal)
else:
    integrity_rows = [{"configuration": cid, "run": 1, "role": "primary",
        "questions": row["planned_n"],
        "missing final-attempt charges": row["operations"]["observed_usd"]["missing_n"],
        "missing call durations": row["operations"]["latency"]["missing_n"],
        "cost handling": row["operations"].get("analysis_usd", {}).get("basis",
            "Unresolved; cost comparison unavailable" if row["operations"]["observed_usd"]["missing_n"] else "Observed billing complete")}
        for cid, row in summary["primary"].items()]

integrity_table = pd.DataFrame([{
    "Model": MODEL_NAMES[r["configuration"]],
    **{k: v for k, v in r.items() if k not in ("configuration", "missing-charge questions")}
} for r in integrity_rows])
issues = integrity_table[(integrity_table["missing final-attempt charges"] > 0) |
                         (integrity_table["missing call durations"] > 0) |
                         (integrity_table.get("missing outcomes", pd.Series(0, index=integrity_table.index)) > 0)]
if len(issues):
    display(Markdown("**Resource integrity flags.** Missing measurements or outcomes are listed below. Review their handling before interpreting resource comparisons."))
    display(issues)
else:
    display(Markdown("All final-attempt charges and call durations are present."))
if SHOW_DETAILS:
    display(integrity_table)


In [ ]:
# This notebook presents the authenticated public numerical reference.
# Optional recalculation uses analysis.public_release with locally reconstructed MedQA keys.
families = summary["paired"]["families"]


## 3. Primary accuracy

All 1,273 questions contribute to each complete configuration's first-valid accuracy. Exhausted failures count as incorrect. S1 contains 195 decision–generative contrasts and 78 ordered decision-pair contrasts, with one Holm adjustment across all 273. S2 contains the three native-effort contrasts with separate Holm adjustment. Intervals remain pointwise. Missing outcomes are not scored as failures.


In [ ]:
accuracy_table = pd.DataFrame([{
    "ID": cid, "Model": MODEL_NAMES[cid], "Status": "Complete" if r["complete"] else "INCOMPLETE",
    "Correct": r["correct"], "n": r["planned_n"], "Observed outcomes": r["terminal_n"],
    "Accuracy (%)": None if r["accuracy"] is None else 100*r["accuracy"],
    "Wilson 95% interval (%)": ci(r["wilson95"], 100),
    "Failed resolved outcomes": r["terminal_n"]-r["valid_n"],
} for cid, r in summary["primary"].items()]).sort_values("Accuracy (%)", ascending=False)
display(accuracy_table.style.format(precision=2, na_rep="Unavailable").hide(axis="index"))
families = summary["paired"].get("families", {})
primary_paired = paired_table(families.get("S1", []))
if families:
    display(primary_paired.style.format(precision=3, na_rep="Unavailable").hide(axis="index"))
else:
    display(Markdown("**S1 unavailable:** " + summary["paired"]["gate"]))


## 4. Resource trade-offs

Cost and latency use the valid initial attempt, first valid retry, or last terminal attempt after exhausted recovery. Each question contributes one charge and one request-wall duration. Earlier calls remain in operational summaries. Primary missing resources use the unrounded observed mean within the same configuration's primary run, including zero values and failed final attempts. A donor-free measure remains unavailable. Repeat runs have observed-only summaries.

The table retains observed and imputed counts. Frontiers use mean final-attempt charge and median final-attempt duration; observed-only frontiers remain in `summary`. Collection campaigns, segments, dates, concurrency and pacing remain separate metadata. Decision models use orange and generative configurations teal. Decimal labels use periods.


In [ ]:
resource_rows = []
for cid, r in summary["primary"].items():
    op = r["operations"]
    cost = op.get("analysis_usd", op["observed_usd"])
    latency = op.get("analysis_latency", op["latency"])
    resource_rows.append({
        "Model": MODEL_NAMES[cid], "Accuracy (%)": None if r["accuracy"] is None else 100*r["accuracy"],
        "Charge (USD/1,000 questions)": None if cost["mean_per_attempt"] is None else 1000*cost["mean_per_attempt"],
        "Cost basis": cost.get("basis", "Observed final-attempt billing"),
        "Missing observed charges": op["observed_usd"]["missing_n"], "Imputed charges": cost.get("imputed_n", 0),
        "Median call duration (s)": latency["median"], "IQR (s)": latency["iqr"],
        "p95 (s)": latency["p95"], "Missing observed durations": op["latency"]["missing_n"],
        "Imputed durations": latency.get("imputed_n", 0),
        "Duration basis": latency.get("basis", "Observed selected-final-attempt durations"),
        "Retry calls": op["retry_calls_n"],
        "Observed cost subtotal (USD)": op["observed_usd"]["observed_subtotal"],
        "Observed duration median (s)": op["latency"]["median"],
        "Observed duration sum (s)": op["latency"]["observed_sum_seconds"],
        "Analysis duration sum (s)": latency["total_seconds"],
        "Cost donor n": cost.get("imputation", {}).get("donor_n"),
        "Duration donor n": latency.get("imputation", {}).get("donor_n"),
    })
resource_table = pd.DataFrame(resource_rows).sort_values("Accuracy (%)", ascending=False)
display(resource_table.style.format(precision=3, na_rep="Unavailable").hide(axis="index"))
basis = "analysis_usd" if "analysis_usd" in summary["frontiers"] else "observed_usd"
for title, frontier in (("Cost comparison", summary["frontiers"][basis]["accuracy_cost"]),
                        ("Duration comparison", summary["frontiers"]["accuracy_latency"])):
    exclusions = [{"Model": MODEL_NAMES[cid], "Unavailable coordinate": ", ".join(reason["missing_coordinates"])}
                  for cid, reason in frontier["excluded"].items()]
    if exclusions:
        display(Markdown(f"**{title}: omitted points**")); display(pd.DataFrame(exclusions))


### Figure 1 and interactive supplement

Figure 1 contains a branched study schematic (a), all 28 accuracy bars with Wilson intervals (b), and cost and duration projections (c, d). Bars start at zero. Each resource panel names its existing Pareto-frontier configurations in full; other short IDs appear where legible and map to the accuracy bars. Cost and duration use the same saved selected-final-attempt estimates.

The separate interactive supplement shows the saved 3D frontier as diamonds and a translucent triangulated mesh. The mesh interpolates between nondominated points in log-cost/log-duration space; it does not represent measurements of intermediate configurations.

Rerun the settings, drawing and supplementary HTML cells after saving. Tests and notebook execution remain manual.


In [ ]:
from copy import deepcopy
from importlib import reload
import analysis.support.figures as figure_helpers
reload(figure_helpers)
FIGURE_SETTINGS = deepcopy(figure_helpers.FIGURE_SETTINGS)
FIGURE_SETTINGS.update(
    report_layout="coauthor-20261021",
    font="Arial",
    # Main Figures 1-3: portrait DIN A4; Extended Data keeps its reviewed dimensions.
    revision_sizes={**figure_helpers.FIGURE_SETTINGS["revision_sizes"]},
    revision_fontsize=9,
    revision_label_fontsize=8,
    revision_legend_columns=2,
    revision_auroc_in_legend=False,  # AUROCs and intervals appear inside each ROC panel and in the table.
    revision_3d_view=(35, -135),  # Elevation, azimuth; higher view from the low-cost/short-duration side.
    resource_label_offsets={},  # Example: {"accuracy_cost": {"D01": (12, -12)}}.
)


In [ ]:
if DRAW_FIGURES and MODE != "compute":
    from analysis.support.figures import build_figures, current_report_figures, report_plot_values
    from analysis.support.source_record import presentation_code_record
    from matplotlib import pyplot as plt
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
    for old_figure in figures.values():
        plt.close(old_figure)
    figures = {}  # Do not retain stale previews after a failed rebuild.
    presentation_source_before = presentation_code_record(REPOSITORY)
    revision = FIGURE_SETTINGS["report_layout"] == "coauthor-20261021"
    visible_figures = current_report_figures(summary, secondary=SHOW_SECONDARY, revision=revision)
    figures, figure_review_record = build_figures(
        summary, plot_data=saved_result["figure_data"], settings=FIGURE_SETTINGS,
        model_names=MODEL_NAMES, figure_names=visible_figures)
    expected = report_plot_values(summary, saved_result["figure_data"], revision=revision)
    recorded = {name: record["plotted_values"] for name, record in figure_review_record["figures"].items()}
    if recorded != {name: expected[name] for name in visible_figures}:
        raise ValueError("Preview values differ from the saved release")
    if presentation_code_record(REPOSITORY) != presentation_source_before:
        raise ValueError("Presentation source changed while drawing; save edits and rerun")
    for figure in figures.values():
        plt.close(figure)
    for name in ("combined_candidate_a", "accuracy", "resource_panel"):
        if name in figures:
            display(Markdown("**" + name.replace("_", " ").capitalize() + "**"))
            display(figures[name])
    for issue in figure_review_record.get("layout_review", []):
        display(Markdown("Layout review: " + issue))


### Interactive supplementary resource figure
Rotate, zoom and hover over the saved model estimates. Diamonds and the translucent triangulated surface identify the 3D Pareto frontier. IDs are off by default. Use **Download HTML** for an offline copy.


In [ ]:
# Supplementary HTML uses saved coordinates independently of the static export roster.
if DRAW_FIGURES and MODE != "compute" and revision:
    from analysis.support.figures import interactive_resource_view_v3, presentation_plot_data
    resource_data = presentation_plot_data(
        summary, saved_result["figure_data"], confidence_display="roc", revision=True)
    display(interactive_resource_view_v3(
        resource_data["resource_3d"]["accuracy_cost_duration"],
        MODEL_NAMES, FIGURE_SETTINGS))


## 5. Secondary confidence

D01–D13 use the same four-label, finite-value, [0,1], positive-sum and inclusive 0.02 sum-deviation rules. Native field meaning must be reviewed. Each model reports its eligible denominator, exclusions, native score support, stratified AUROC interval, ten reliability bins with Wilson intervals, Brier score, high-confidence errors and whole-tie risk–coverage. All displays retain model-specific denominators.

Figures 2 and 3 display the six decision models on the **saved joint accuracy–cost–duration Pareto frontier** among all 28 configurations: JEV, GPT-6 Luna Decisions, Mercury Decide, Liquid d1, Decider V1.1 27B and Clef Omni. Selection uses primary-run accuracy, mean selected-final-attempt cost and median duration on the observed-plus-imputed resource basis. A configuration is dominated when another is at least as accurate and uses no more of either resource, with at least one strict improvement. This descriptive selection uses the saved three-dimensional frontier, not separate two-dimensional frontiers or probability-performance rankings.

Both main figures use six model rows and two columns, with each model’s plots paired horizontally. Model order is JEV, GPT-6 Luna Decisions, Mercury Decide, Liquid d1, Decider V1.1 27B and Clef Omni. Every panel has identical square plotting dimensions and its own x- and y-axis headings and tick labels. Bold 12-point panel letters a–l appear at the upper-left corner of each panel, offset 40 points left of the plotting area. Titles and axis headings use 11-point text, ticks 10-point text, and AUROC annotations 9.5-point text. Eligible counts remain in the adjacent text and table. **Figure 2** pairs error versus eligible coverage with error versus inclusive probability cutoff; every error axis spans 0–25% with ticks at 0%, 10%, 20% and 25%. Both figures use 1.3-point plot lines, including ROC curves, reference diagonals and calibration interval marks. Curves retain whole score ties; values outside the displayed range are clipped without changing saved estimates. Cutoffs retaining no answers have undefined error and no plotted estimate. **Figure 3** pairs ROC with calibration, with both axes ticked at 0, 0.25, 0.5, 0.75 and 1. Every ROC panel reports its saved AUROC and stratified-bootstrap 95% interval. Calibration shows the ten saved bins as unconnected points with Wilson 95% intervals; empty bins have no estimate.


In [ ]:
if SHOW_SECONDARY:
    confidence_rows = []
    for cid, model in summary["decision_models"].items():
        c = model["confidence"]; h = c["high_confidence"]
        confidence_rows.append({"Model": MODEL_NAMES[cid], "Primary run complete": c["complete_primary_run"], "Eligible n": c["eligible_n"], "Planned n": c["planned_n"],
            "Exclusions": c["exclusions"], "Field review": model.get("probability_semantics_verified"),
            "AUROC": c["auroc"], "AUROC 95% interval": ci(c["auroc_bootstrap"]["interval95"]),
            "Brier": c["binary_brier"], "Errors at score ≥0.90": h["errors"],
            "Answers at score ≥0.90": h["n"], "Error rate at score ≥0.90 (%)": 100*h["error_rate"] if h["error_rate"] is not None else None})
    display(pd.DataFrame(confidence_rows).style.format(precision=3, na_rep="Unavailable").hide(axis="index"))
    for title, name in (("Figure 2. Error versus coverage and probability cutoff", "probability_errors"),
                        ("Figure 3. ROC and calibration", "probability_roc_calibration")):
        display(Markdown("**" + title + "**"))
        if name in figures:
            display(figures[name])
    display(Markdown("**Supplementary probability figures: all 13 decision models.**"))
    for title, name in (("ROC", "probability_roc_supplement"),
                        ("Calibration", "probability_calibration"),
                        ("Error versus coverage", "probability_coverage"),
                        ("Error versus probability cutoff", "probability_cutoff_supplement")):
        display(Markdown("**Supplement: " + title + "**"))
        if name in figures:
            display(figures[name])


## 6. Repeatability and native effort

Every decision model has three separate runs. The shared six-panel figure shows runs 1–2, 1–3 and 2–3 in columns, with own-selected-answer probability change above and normalized-vector total variation below. Colors and line patterns match the confidence figure. Each legend entry reports eligible matched questions; a curve farther left indicates smaller changes.

The tables report per-run accuracy, valid-label agreement, correctness/failure transitions, own-selected and earlier-selected probability changes, normalized total variation, and exact three-run selected-probability/vector agreement. Agreement and unchanged rates are fractions (0–1). Exact equality is evaluated before display rounding. Stable answers can still be wrong. Native-effort contrasts follow the repeatability tables.

“Unavailable” means the statistic cannot be calculated on an eligible sample. Each probability measure retains its own denominator and exclusions. Empty reliability bins and high-confidence groups with no answers have undefined rates.


In [ ]:
if SHOW_SECONDARY:
    for title, rows in repeatability_tables(summary, MODEL_NAMES).items():
        if rows:
            display(Markdown(f"**{title}**"))
            display(pd.DataFrame(rows).style.format(precision=4, na_rep="Unavailable").hide(axis="index"))
    for name in ("repeatability_supplement", "repeatability_panel"):
        if name in figures:
            display(Markdown("**Supplementary repeatability candidate**"))
            display(figures[name])
    display(Markdown("Columns compare runs 1–2, 1–3 and 2–3. Panels a–c show the absolute "
        "difference between each run’s own selected-answer probability; the answer labels may differ. "
        "Panels d–f show total variation across the four normalized option probabilities. "
        "Curves give the cumulative percentage of eligible matched questions at or below each change. "
        "A probability difference of 0.10 is ten percentage points. Table 5 and the main-text "
        "interpretation remain in the paper; this complete six-panel display is intended for the supplement."))
    if "repeatability_supplement" in figures:
        payload = figure_review_record["figures"]["repeatability_supplement"]["plotted_values"]
        display(pd.DataFrame([{"Model": MODEL_NAMES[r["configuration"]],
            "Measure": "Own-selected probability" if key == "repeatability_confidence" else "Whole-vector total variation",
            "Run pair": str(r["earlier_run"])+"–"+str(r["later_run"]), "Eligible n": r["eligible_n"]}
            for key, rows in payload.items() for r in rows]))
    if families: display(paired_table(families["S2"]))


## 7. Sensitivities

Original first-attempt accuracy shows the effect of the post-collection recovery policy. The visual sensitivity excludes the fixed, outcome-informed 108 nominations and retains 1,165 questions. These nominations concern potentially missing visual material; answerability is not adjudicated. Paired complete-valid analyses condition on usable answers from both models. Sensitivities add no hypothesis-test family.


In [ ]:
sensitivity_rows = []
for cid, r in summary["primary"].items():
    original = summary["first_attempt_sensitivity"][cid]
    visual = summary["missing_visual_sensitivity"].get("accuracy", {}).get(cid, {})
    sensitivity_rows.append({"Model": MODEL_NAMES[cid], "Primary n": r["planned_n"],
        "Primary accuracy": r["accuracy"], "First-attempt accuracy": original["accuracy"],
        "Visual sensitivity n": visual.get("n"), "Visual sensitivity accuracy": visual.get("accuracy")})
display(pd.DataFrame(sensitivity_rows))
if SHOW_DETAILS:
    for name in ("complete_valid_sensitivity", "missing_visual_sensitivity"):
        sensitivity = summary.get(name, {})
        display(Markdown(f"**{name}**"))
        for rows in sensitivity.get("paired", sensitivity).get("families", {}).values(): display(paired_table(rows))
    for cid, model in summary["decision_models"].items():
        display(Markdown(f"**{MODEL_NAMES[cid]}: reliability bins**"))
        display(pd.DataFrame(model["confidence"]["reliability"] or []))


## 8. Export the reproduced presentation

Exports are enabled in PDF, SVG and 300-dpi PNG. Run All creates a fresh `analysis/output/public-render-v01/`, with the reproduced graphics, copies of the verified public table projections and a new provenance manifest. Existing destinations are refused. The table copies are identified as saved projections; full statistical recalculation is a separate explicit README command.


In [ ]:
EXPORT_REPORT = True
EXPORT_DIR = REPOSITORY / "analysis/output/public-render-v01"
EXPORT_FORMATS = ("pdf", "svg", "png")
EXPORT_DPI = 300
if EXPORT_REPORT:
    exported = export_public_report(EXPORT_DIR, figures, figure_review_record, root=REPOSITORY, source_before=presentation_source_before)
    print("Exported public presentation:", exported)
